# AlzheimerCNN Ablation Study

This notebook is the ablation study for the `AlzheimerCNN` model contribution to PyHealth, focused on reproducing the ablations and results from the original paper.

**Reference paper:** Liu et al. *"On the design of convolutional neural networks for automatic detection of Alzheimer's disease."* ML4H @ NeurIPS 2019. https://arxiv.org/abs/1911.03740

## Experimental Setup

We train five model configurations on the `Falah/Alzheimer_MRI` dataset and compare validation accuracy and macro-F1. All models share the same training recipe:

- **Optimizer:** AdamW, lr=1e-4, weight_decay=1e-5
- **Epochs:** 50
- **Batch size:** 32
- **Train/Val/Test split:** 60% / 20% / 20%
- **Loss:** Cross-entropy (4-class classification)

## Configurations Tested

| # | Model | Variation | Hypothesis |
|---|-------|-----------|------------|
| 1 | `GenericCNN` | `num_layers = 8, k = 3, norm_type = "batch"` (baseline) | Reference point. Larger layers with gradual widening of channels, kernel size = 3, batch normalization |
| 2 | `GenericCNN` | `num_layers = 4` | Less # of layers with faster widening of channels greatly improve performance and barely hurt metrics |
| 3 | `GenericCNN` | `k = 1, 3, 5, 3` | Starting with smaller kernels greatly improves metrics as tiny details matter |
| 4 | `GenericCNN` | `norm_type="instance"` | Instance normalization preferred to keep images independent |


## 1. Setup


In [1]:
import torch
import pandas as pd

from pyhealth.datasets import split_by_sample, get_dataloader
from pyhealth.trainer import Trainer

# ── Auto-detect device ──────────────────────────────────────────────────
DEVICE = "cpu"
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"

print(f"Using device: {DEVICE}")

METRICS = ["accuracy", "f1_macro", "balanced_accuracy"]
EPOCHS = 50
BATCH_SIZE = 32
SEED = 42

torch.manual_seed(SEED)


/usr/local/lib/python3.12/dist-packages/pyhealth/trainer.py:12: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import trange


Using device: cuda


<font size="8">Load Models</font>

In [2]:
from typing import Dict, List, Optional
import torch
import torch.nn as nn
import torch.nn.functional as F
from pyhealth.datasets import SampleDataset
from pyhealth.models.base_model import BaseModel

# ── Swap ACTIVE_CONFIG to switch between the four variants ────────────────────
CONFIGS = {
    "v1_baseline": {
        "num_layers": 8,
        "kernel_sizes": None,       # None = all-3 kernels
        "norm_type": "batch",
    },
    "v2_fewer_layers": {
        "num_layers": 4,
        "kernel_sizes": None,
        "norm_type": "batch",
    },
    "v3_mixed_kernels": {
        "num_layers": 4,
        "kernel_sizes": [1, 3, 5, 3],
        "norm_type": "batch",
    },
    "v4_instance_norm": {
        "num_layers": 4,
        "kernel_sizes": [1, 3, 5, 3],
        "norm_type": "instance",
    },
}

ACTIVE_CONFIG = "v1_baseline"


def _make_norm(norm_type: str, num_channels: int) -> nn.Module:
    if norm_type == "batch":
        return nn.BatchNorm2d(num_channels)
    if norm_type == "instance":
        return nn.InstanceNorm2d(num_channels)
    raise ValueError(f"Unknown norm_type: {norm_type!r}")


class GenericCNN(BaseModel):
    def __init__(
        self,
        dataset: SampleDataset,
        init_channels: int = 32,
        classifier_hidden_dim: int = 128,
        dropout: float = 0.5,
        config_name: str = ACTIVE_CONFIG,
        **kwargs,
    ) -> None:
        super(GenericCNN, self).__init__(dataset=dataset)

        cfg = CONFIGS[config_name]
        num_layers: int       = cfg["num_layers"]
        kernel_sizes: Optional[List[int]] = cfg["kernel_sizes"]
        norm_type: str        = cfg["norm_type"]

        # Resolve kernel sizes: explicit list or all-3s
        if kernel_sizes is None:
            kernel_sizes = [3] * num_layers
        assert len(kernel_sizes) == num_layers, (
            f"kernel_sizes length {len(kernel_sizes)} must match num_layers {num_layers}"
        )

        self.init_channels = init_channels
        self.classifier_hidden_dim = classifier_hidden_dim

        # Build convolutional blocks dynamically
        # Channel widening: doubles every block, capped at init_channels * 8
        # (fast for 4 layers, gradual for 8 layers — driven purely by num_layers)
        blocks: List[nn.Module] = []
        in_ch = 1
        for i, k in enumerate(kernel_sizes):
            # For num_layers=4 channels go 1→C→2C→4C→8C (fast widening)
            # For num_layers=8 channels go 1→C→C→2C→2C→4C→4C→8C→8C (gradual)
            out_ch = init_channels * (2 ** (i * 4 // num_layers))

            pool = (
                nn.AdaptiveAvgPool2d((1, 1))
                if i == num_layers - 1
                else nn.MaxPool2d(kernel_size=2, stride=2)
            )
            blocks.append(nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=k, stride=1, padding=k // 2),
                _make_norm(norm_type, out_ch),
                nn.LeakyReLU(inplace=True),
                pool,
            ))
            in_ch = out_ch

        self.blocks = nn.ModuleList(blocks)
        final_channels = in_ch

        # Classifier Head
        output_size = self.get_output_size()
        self.classifier = nn.Sequential(
            nn.Linear(final_channels, classifier_hidden_dim),
            nn.LeakyReLU(inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(classifier_hidden_dim, output_size),
        )

    def forward(self, **kwargs) -> Dict[str, torch.Tensor]:
        x      = kwargs[self.feature_keys[0]].to(self.device)
        labels = kwargs[self.label_keys[0]].to(self.device)

        for block in self.blocks:
            x = block(x)

        emb    = torch.flatten(x, 1)
        logits = self.classifier(emb)
        loss   = self.get_loss_function()(logits, labels)
        y_prob = self.prepare_y_prob(logits)

        return {
            "loss":   loss,
            "y_prob": y_prob,
            "y_true": labels,
            "logit":  logits,
        }

## 2. Load Dataset

Uses the `AlzheimersDataset` PyHealth-compatible wrapper around `Falah/Alzheimer_MRI` (4-class severity classification: NonDemented, VeryMildDemented, MildDemented, ModerateDemented).

The dataset class auto-downloads from HuggingFace and saves a `alzheimers-metadata.csv` for PyHealth's BaseDataset to consume.


In [3]:
from typing import Any, Dict, List
from pyhealth.tasks.base_task import BaseTask


class AlzheimersClassification(BaseTask):
    """Task for classifying Alzheimer's disease from brain images."""

    task_name: str = "AlzheimersClassification"
    input_schema: Dict = {"image": ("image", {"mode": "L", "image_size": 128})}
    output_schema: Dict[str, str] = {"label": "multiclass"}

    def __call__(self, patient: Any) -> List[Dict[str, Any]]:
        events = patient.get_events(event_type="alzheimers")
        assert len(events) == 1, f"Expected 1 image, got {len(events)}"
        event = events[0]
        return [{"image": event.path, "label": event.label}]

In [6]:
import logging
import os
from pathlib import Path
from typing import Optional

import pandas as pd
from datasets import load_dataset
from pyhealth.datasets.base_dataset import BaseDataset

logger = logging.getLogger(__name__)


class AlzheimersDataset(BaseDataset):
    """PyHealth-compatible Alzheimer's dataset (auto-download + process)."""

    def __init__(
        self,
        root: str,
        dataset_name: Optional[str] = None,
        config_path: Optional[str] = None,
        cache_dir: Optional[str] = None,
        num_workers: int = 1,
        dev: bool = False,
        hf_dataset_name: str = "Falah/Alzheimer_MRI",
    ) -> None:
        self.hf_dataset_name = hf_dataset_name

        try:
            base_path = Path(__file__).parent
        except NameError:
            base_path = Path.cwd()

        if config_path is None:
            config_path = base_path / "configs" / "alzheimers.yaml"

        metadata_path = os.path.join(root, "alzheimers-metadata.csv")

        if not os.path.exists(metadata_path):
            self.prepare_metadata(root)

        super().__init__(
            root=root,
            tables=["alzheimers"],
            dataset_name=dataset_name or "alzheimers",
            config_path=config_path,
            cache_dir=cache_dir,
            num_workers=num_workers,
            dev=dev,
        )

    def prepare_metadata(self, root: str) -> None:
        os.makedirs(root, exist_ok=True)
        image_dir = os.path.join(root, "images")
        os.makedirs(image_dir, exist_ok=True)
        metadata_path = os.path.join(root, "alzheimers-metadata.csv")

        ds = load_dataset(self.hf_dataset_name, split="train")
        records = []
        for i, item in enumerate(ds):
            img_path = os.path.join(image_dir, f"{i}.png")
            if not os.path.isfile(img_path):
                item["image"].save(img_path)
            records.append({
                "patient_id": str(i),
                "path": img_path,
                "label": str(item["label"]),
            })
        pd.DataFrame(records).to_csv(metadata_path, index=False)

In [7]:
# Instantiate dataset and create train/val/test splits
dataset = AlzheimersDataset(root="Downloads")
dataset_samples = dataset.set_task(AlzheimersClassification(), num_workers=1)

train_samples, val_samples, test_samples = split_by_sample(
    dataset_samples, [0.6, 0.2, 0.2]
)

train_loader = get_dataloader(train_samples, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = get_dataloader(val_samples,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = get_dataloader(test_samples,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Train: {len(train_samples)} | Val: {len(val_samples)} | Test: {len(test_samples)}")

Initializing alzheimers dataset from Downloads (dev mode: False)


INFO:pyhealth.datasets.base_dataset:Initializing alzheimers dataset from Downloads (dev mode: False)


No cache_dir provided. Using default cache dir: /root/.cache/pyhealth/5e3d2e44-3165-596e-89d3-ed772a0ed564


INFO:pyhealth.datasets.base_dataset:No cache_dir provided. Using default cache dir: /root/.cache/pyhealth/5e3d2e44-3165-596e-89d3-ed772a0ed564


Setting task AlzheimersClassification for alzheimers base dataset...


...

Worker 0 finished processing samples.



INFO:pyhealth.datasets.base_dataset:Worker 0 finished processing samples.


Cached processed samples to /root/.cache/pyhealth/5e3d2e44-3165-596e-89d3-ed772a0ed564/tasks/AlzheimersClassification_7e906453-ea6b-5cc3-8506-3e00427d79ba/samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld


INFO:pyhealth.datasets.base_dataset:Cached processed samples to /root/.cache/pyhealth/5e3d2e44-3165-596e-89d3-ed772a0ed564/tasks/AlzheimersClassification_7e906453-ea6b-5cc3-8506-3e00427d79ba/samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld


Train: 3072 | Val: 1024 | Test: 1024


## 3. Training Helper

A single function that trains any model with the shared recipe and returns final validation metrics. This keeps every experiment cell short and ensures all configurations are compared fairly.


In [8]:
import time
def train_and_evaluate(model, name: str) -> dict:
    """Train a model and return its final validation metrics.

    Args:
        model: An instantiated PyHealth model inheriting from BaseModel.
        name: Human-readable label for the configuration.

    Returns:
        Dict with keys 'name', 'accuracy', 'f1_macro', 'balanced_accuracy', 'loss', 'params'.
    """
    print(f"\n{'='*60}")
    print(f"Training: {name}")
    print(f"{'='*60}")

    trainer = Trainer(model=model, metrics=METRICS, device=DEVICE)

    t0 = time.perf_counter()
    trainer.train(
        train_dataloader=train_loader,
        val_dataloader=val_loader,
        epochs=EPOCHS,
        optimizer_class=torch.optim.AdamW,
        optimizer_params={"lr": 1e-4, "weight_decay": 1e-5},
    )
    train_time = time.perf_counter() - t0

    scores = trainer.evaluate(val_loader)
    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    return {
        "name": name,
        "accuracy": scores.get("accuracy", float("nan")),
        "f1_macro": scores.get("f1_macro", float("nan")),
        "balanced_accuracy": scores.get("balanced_accuracy", float("nan")),
        "loss": scores.get("loss", float("nan")),
        "params": num_params,
        "train_time_seconds": round(train_time, 1),
    }


# Collect all results in this list for the final summary
results = []


## 4. Experiments


### Experiment 1: Baseline (`init_channels=32`)


In [9]:
model = GenericCNN(
    dataset=train_samples,
    init_channels=16,
    classifier_hidden_dim=128,
    dropout=0.5,
    config_name="v1_baseline",
)
results.append(train_and_evaluate(model, "CNN (8 layers, k=3, batch norm)"))



Training: CNN (8 layers, k=3, batch norm)
GenericCNN(
  (blocks): ModuleList(
    (0): Sequential(
      (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): LeakyReLU(negative_slope=0.01, inplace=True)
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (1): Sequential(
      (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): LeakyReLU(negative_slope=0.01, inplace=True)
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (2): Sequential(
      (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): LeakyReLU(negative_slope=0.01, inplace=True)
   

INFO:pyhealth.trainer:GenericCNN(
  (blocks): ModuleList(
    (0): Sequential(
      (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): LeakyReLU(negative_slope=0.01, inplace=True)
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (1): Sequential(
      (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): LeakyReLU(negative_slope=0.01, inplace=True)
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (2): Sequential(
      (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): LeakyReLU(negative_slope=0.01, inplace=True)
      (3): MaxPool2d(ker

Metrics: ['accuracy', 'f1_macro', 'balanced_accuracy']


INFO:pyhealth.trainer:Metrics: ['accuracy', 'f1_macro', 'balanced_accuracy']


Device: cuda


...

INFO:pyhealth.trainer:f1_macro: 0.7551


balanced_accuracy: 0.7167


INFO:pyhealth.trainer:balanced_accuracy: 0.7167


loss: 1.0061


INFO:pyhealth.trainer:loss: 1.0061
Evaluation: 100%|██████████| 32/32 [00:00<00:00, 101.45it/s]


### Experiment 2: Fewer CNN layers

In [10]:
model = GenericCNN(
    dataset=train_samples,
    init_channels=16,
    classifier_hidden_dim=128,
    dropout=0.5,
    config_name="v2_fewer_layers",
)
results.append(train_and_evaluate(model, "CNN (4 layers, k = 3, batch norm)"))



Training: CNN (4 layers, k = 3, batch norm)
GenericCNN(
  (blocks): ModuleList(
    (0): Sequential(
      (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): LeakyReLU(negative_slope=0.01, inplace=True)
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (1): Sequential(
      (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): LeakyReLU(negative_slope=0.01, inplace=True)
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (2): Sequential(
      (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): LeakyReLU(negative_slope=0.01, inplace=True)
 

INFO:pyhealth.trainer:GenericCNN(
  (blocks): ModuleList(
    (0): Sequential(
      (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): LeakyReLU(negative_slope=0.01, inplace=True)
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (1): Sequential(
      (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): LeakyReLU(negative_slope=0.01, inplace=True)
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (2): Sequential(
      (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): LeakyReLU(negative_slope=0.01, inplace=True)
      (3): MaxPool2d(ker

Metrics: ['accuracy', 'f1_macro', 'balanced_accuracy']


INFO:pyhealth.trainer:Metrics: ['accuracy', 'f1_macro', 'balanced_accuracy']


Device: cuda


...

INFO:pyhealth.trainer:f1_macro: 0.3400


balanced_accuracy: 0.4410


INFO:pyhealth.trainer:balanced_accuracy: 0.4410


loss: 2.4650


INFO:pyhealth.trainer:loss: 2.4650
Evaluation: 100%|██████████| 32/32 [00:00<00:00, 103.18it/s]


### Experiment 3: Kernel sizes starting small


In [11]:
model = GenericCNN(
    dataset=train_samples,
    init_channels=16,
    classifier_hidden_dim=128,
    dropout=0.5,
    config_name="v3_mixed_kernels",
)
results.append(train_and_evaluate(model, "CNN (4 layers, k=1,3,5,3, batch norm)"))



Training: CNN (4 layers, k=1,3,5,3, batch norm)
GenericCNN(
  (blocks): ModuleList(
    (0): Sequential(
      (0): Conv2d(1, 16, kernel_size=(1, 1), stride=(1, 1))
      (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): LeakyReLU(negative_slope=0.01, inplace=True)
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (1): Sequential(
      (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): LeakyReLU(negative_slope=0.01, inplace=True)
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (2): Sequential(
      (0): Conv2d(32, 64, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): LeakyReLU(negative_slope=0.01, inplace=True)
      (3): Ma

INFO:pyhealth.trainer:GenericCNN(
  (blocks): ModuleList(
    (0): Sequential(
      (0): Conv2d(1, 16, kernel_size=(1, 1), stride=(1, 1))
      (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): LeakyReLU(negative_slope=0.01, inplace=True)
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (1): Sequential(
      (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): LeakyReLU(negative_slope=0.01, inplace=True)
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (2): Sequential(
      (0): Conv2d(32, 64, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): LeakyReLU(negative_slope=0.01, inplace=True)
      (3): MaxPool2d(kernel_size=2, stri

Metrics: ['accuracy', 'f1_macro', 'balanced_accuracy']


INFO:pyhealth.trainer:Metrics: ['accuracy', 'f1_macro', 'balanced_accuracy']


Device: cuda


...

INFO:pyhealth.trainer:f1_macro: 0.4979


balanced_accuracy: 0.5019


INFO:pyhealth.trainer:balanced_accuracy: 0.5019


loss: 0.8106


INFO:pyhealth.trainer:loss: 0.8106
Evaluation: 100%|██████████| 32/32 [00:00<00:00, 99.71it/s] 


### Experiment 4: Instance normalization, finalized CNN Model from the paper


In [ ]:
model = GenericCNN(
    dataset=train_samples,
    init_channels=16,
    classifier_hidden_dim=128,
    dropout=0.5,
    config_name="v4_instance_norm",
)
results.append(train_and_evaluate(model, "CNN (4 layers, k = 1, 3, 5, 3, instance norm)"))



Training: CNN (4 layers, k = 1, 2, 3, 4, instance norm)
GenericCNN(
  (blocks): ModuleList(
    (0): Sequential(
      (0): Conv2d(1, 16, kernel_size=(1, 1), stride=(1, 1))
      (1): InstanceNorm2d(16, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
      (2): LeakyReLU(negative_slope=0.01, inplace=True)
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (1): Sequential(
      (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
      (2): LeakyReLU(negative_slope=0.01, inplace=True)
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (2): Sequential(
      (0): Conv2d(32, 64, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
      (1): InstanceNorm2d(64, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
      (2): LeakyReLU(negative_slope=0.01, inpl

INFO:pyhealth.trainer:GenericCNN(
  (blocks): ModuleList(
    (0): Sequential(
      (0): Conv2d(1, 16, kernel_size=(1, 1), stride=(1, 1))
      (1): InstanceNorm2d(16, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
      (2): LeakyReLU(negative_slope=0.01, inplace=True)
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (1): Sequential(
      (0): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
      (2): LeakyReLU(negative_slope=0.01, inplace=True)
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (2): Sequential(
      (0): Conv2d(32, 64, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
      (1): InstanceNorm2d(64, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
      (2): LeakyReLU(negative_slope=0.01, inplace=True)
      (3): MaxPool2d(kern

Metrics: ['accuracy', 'f1_macro', 'balanced_accuracy']


INFO:pyhealth.trainer:Metrics: ['accuracy', 'f1_macro', 'balanced_accuracy']


Device: cuda


...

INFO:pyhealth.trainer:f1_macro: 0.3166


balanced_accuracy: 0.3489


INFO:pyhealth.trainer:balanced_accuracy: 0.3489


loss: 0.8433


INFO:pyhealth.trainer:loss: 0.8433
Evaluation: 100%|██████████| 32/32 [00:00<00:00, 90.48it/s]


## 5. Results Summary

All six configurations side by side, sorted by validation accuracy.


In [13]:
df = pd.DataFrame(results)
df = df[["name", "params", "accuracy", "f1_macro", "balanced_accuracy", "loss", "train_time_seconds"]]
df = df.sort_values("accuracy", ascending=False).reset_index(drop=True)

# Format columns for readability
df["params"]   = df["params"].apply(lambda x: f"{x:,}")
df["accuracy"] = df["accuracy"].apply(lambda x: f"{x:.4f}")
df["f1_macro"] = df["f1_macro"].apply(lambda x: f"{x:.4f}")
df["balanced_accuracy"] = df["balanced_accuracy"].apply(lambda x: f"{x:.4f}")
df["loss"]     = df["loss"].apply(lambda x: f"{x:.4f}")
df["train_time_seconds"].apply(lambda x: f"{x:.4f}")

print(df.to_string(index=False))


                                         name  params accuracy f1_macro balanced_accuracy   loss  train_time_seconds
              CNN (8 layers, k=3, batch norm) 311,220   0.8008   0.7551            0.7167 1.0061               179.4
        CNN (4 layers, k=1,3,5,3, batch norm) 147,300   0.8008   0.4979            0.5019 0.8106               179.9
CNN (4 layers, k = 1, 2, 3, 4, instance norm) 146,820   0.5879   0.3166            0.3489 0.8433               181.8
            CNN (4 layers, k = 3, batch norm) 114,660   0.4453   0.3400            0.4410 2.4650               173.3


## 6. Findings & Discussion

### Headline result
We can conclude from these tests that Experiment 1 performed the best. Each model was very consistent in training time, with only a 8 second difference separating the fastest from the slowest, but the difference in performance between Experiment 1 and the rest was quite drastic. Experiment 3 had only half the parameter and was able to match the accuracy, but the F1 score and balanced accuracy scores were far behind.


### Architecture finding: 8 Layers beats 4 each time.


### Normalization finding: Batch Normalization outperformed Instance Normalization

### Practical recommendation
The recommendation suggested by these sets of experiments is to use 8 CNN layers with a consistent Kernel size of 3x3, as well as using Batch Normalization.

